# 承运商账单核对 · SQL 练习本

> **目标**：不看我写的参考实现，自己用 SQL 把 5 条对账规则写出来。
> 每写完一条，下面的 `✅ 校验` 单元格会自动和参考实现比对，告诉你漏了哪些、多了哪些。

---

## ⚠️ 先确认内核（很重要）

`duckdb` 只装在 **Anaconda** 里，Homebrew 的 Python 没有。所以：

**用这条命令打开 Jupyter**（不要只写 `jupyter lab`，那可能走到 Homebrew 的 Python）：

```bash
cd ~/projects/zurich_insurement_data_eng_job/ai-assisted-reconciliation
/opt/anaconda3/bin/jupyter lab        # 或 notebook
```

如果已经打开了、但第 0 节报 `No module named duckdb`：
菜单 **Kernel → Change kernel → 选 Anaconda 的 python3**，然后 Restart & Run All。

---

## 使用说明

1. **每节的顺序**：读题 → **自己写** → 卡住了看「💡 卡住了再看」→ 跑校验
2. **运行顺序**：从上往下依次运行。**第 0 节必须先跑**（注册视图 + 加载答案）
3. **你要写的**：标了 `✍️` 的单元格 —— 里面是**空的 SQL 骨架**
4. **写错的代价**：没有代价。校验单元格只会告诉你差在哪，随便试
5. **实在写不出**：再看 `sql/reconcile_duckdb.sql`（完整参考实现）

---

## 5 条规则速览

| 规则 | 名称 | 一句话逻辑 |
| --- | --- | --- |
| **R1** | 未入账 | 内部有发货记录，账单中查无此运单 |
| **R2** | 重复计费 | 同一运单在账单中出现 > 1 次 |
| **R3** | 计费重量偏差 | 相对偏差 > 5% 且绝对差 > 0.05kg |
| **R4** | 费率不符 | 分区不一致，或各项费用与费率表不符 |
| **R5** | 金额加总错误 | 行总额 ≠ 各项费用之和 |

> ⚠️ **这里故意不写死「目标条数」** —— 因为条数会随**当前数据集**变化：
> 合成数据（2 万运单）和 Olist 真实数据（11.2 万运单）的结果完全不同。
> **跑完第 0 节后，它会自动打印出当前的目标条数**，以那个为准。
>
> 另外，每个 `✅ 校验` 单元格也会显示 `参考条数 X   你的条数 Y`，直接对照即可。

---

## 0 · 环境准备

注册三张源表视图 + 加载参考结果（答案）。**必须先跑这一节。**

In [2]:
# ⚠️ 必须用装有 duckdb 的内核（Anaconda），否则见最上方「先确认内核」
try:
    import duckdb
except ModuleNotFoundError:
    raise SystemExit(
        '\n❌ 当前内核没有 duckdb。\n'
        '   关闭本 notebook，改用下面命令重新打开：\n'
        '       /opt/anaconda3/bin/jupyter lab\n'
        '   或在菜单 Kernel → Change kernel 里选 Anaconda 的 python3\n'
    )
import pandas as pd
import re

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)

DATA = 'data'
con = duckdb.connect()          # 内存库；DuckDB 直接读 CSV，无需加载步骤

for name, fn in [('internal', 'internal_records.csv'),
                 ('invoice',  'carrier_invoice.csv'),
                 ('rate',     'rate_card.csv')]:
    con.execute(f"CREATE OR REPLACE VIEW {name} AS "
                f"SELECT * FROM read_csv_auto('{DATA}/{fn}')")

REFERENCE = pd.read_csv(f'{DATA}/discrepancies.csv')   # 参考实现的结果（答案）


def _is_blank(sql: str) -> bool:
    """判断练习单元格是否还没写实质内容（只有注释也算没写）。

    要同时认两种注释风格，否则 /* ... */ 会被当成真 SQL 去执行：
        -- 行注释
        /* 块注释 */
    """
    if not sql:
        return True
    # s = re.sub(r'/\*.*?\*/', '', sql, flags=re.S)   # 去掉块注释
    s = re.sub(r'--[^\n]*', '', sql)                    # 去掉行注释
    return not s.strip()

def run(sql: str) -> pd.DataFrame:
    """执行 SQL，结果以 DataFrame 返回。"""
    return con.execute(sql).df()

def check(rule: str, your_df, id_col: str = 'shipment_id') -> bool:
    """与参考实现比对，打印漏报 / 多报。"""
    if your_df is None:
        print('⏳ 还没写 SQL，跳过校验'); return False
    exp = set(REFERENCE.loc[REFERENCE['rule'] == rule, id_col].astype(str))
    got = set(your_df[id_col].astype(str))
    miss, extra = exp - got, got - exp
    print(f'参考条数 {len(exp):>5}   你的条数 {len(got):>5}')
    print(f'  漏报 {len(miss):>5}   多报 {len(extra):>5}')
    if miss:  print(f'  漏报示例：{sorted(miss)[:3]}')
    if extra: print(f'  多报示例：{sorted(extra)[:3]}')
    ok = not miss and not extra
    print('\n✅ 完全一致，这条过了' if ok else '\n❌ 还不一致，继续调')
    return ok

print('✅ 环境就绪')
print(f'   duckdb {duckdb.__version__}   参考结果 {len(REFERENCE):,} 条')

# 打印当前数据集下、各规则的参考条数（换数据集后会自动跟着变）
print()
print('当前数据集的目标条数：')
_rules = ['R1_MISSING_FROM_INVOICE', 'R2_DUPLICATE_BILLING', 'R3_WEIGHT_DEVIATION',
          'R4_RATE_MISMATCH', 'R5_AMOUNT_MISMATCH']
_labels = ['R1 未入账', 'R2 重复计费', 'R3 计费重量偏差', 'R4 费率不符', 'R5 金额加总错误']
for _lab, _r in zip(_labels, _rules):
    print(f'   {_lab:<16}{int((REFERENCE["rule"] == _r).sum()):>6,}  条')
print(f'   {"合计":<16}{len(REFERENCE):>6,}  条')


✅ 环境就绪
   duckdb 1.5.5   参考结果 2,503 条


### 先看一眼三张表长什么样（不用写，直接跑）

In [3]:
print('=== internal_records（内部真相：应付分区 + 计费重量）===')
display(run('SELECT * FROM internal LIMIT 3'))
print()
print('=== carrier_invoice（承运商账单：被核对方）===')
display(run('SELECT * FROM invoice LIMIT 3'))
print()
print('=== rate_card（费率表：把「分区+重量」翻译成金额）===')
display(run("SELECT * FROM rate WHERE zone=6 AND service_level='express' ORDER BY weight_min_kg"))

=== internal_records（内部真相：应付分区 + 计费重量）===


,shipment_id,order_id,seller_id,ship_date,dest_country,dest_state,dest_zip_prefix,dest_zone,service_level,actual_weight_g,length_cm,width_cm,height_cm,volumetric_weight_kg,chargeable_weight_kg,declared_value,currency
0,SHP2026000001,ORD19116100001,SEL0303,2026-07-25,AU,-,14165,8,economy,262,11.3,8.3,7.5,0.117,0.262,16.77,USD
1,SHP2026000002,ORD71788900002,SEL0143,2026-06-01,US,-,30926,5,standard,239,10.8,7.8,4.3,0.060,0.239,280.77,USD
2,SHP2026000003,ORD73305200003,SEL0136,2026-06-06,GB,-,70217,6,standard,5356,18.6,14.4,17.5,0.781,5.356,216.81,USD



=== carrier_invoice（承运商账单：被核对方）===


,invoice_line_id,invoice_batch,shipment_id,invoice_date,billed_zone,service_level,billed_weight_kg,base_amount,weight_amount,fuel_surcharge,remote_surcharge,total_amount,currency
0,L0000001,INV-202609,SHP2026000001,2026-08-04,8,economy,0.262,3.5275,0.6904,0.5483,1.87,6.6362,USD
1,L0000002,INV-202609,SHP2026000002,2026-06-12,5,standard,0.239,3.2000,0.4900,0.5535,0.00,4.2435,USD
2,L0000003,INV-202609,SHP2026000003,2026-06-12,6,standard,5.356,3.4500,10.0693,2.0279,0.00,15.5472,USD



=== rate_card（费率表：把「分区+重量」翻译成金额）===


,zone,service_level,weight_min_kg,weight_max_kg,base_rate,per_kg_rate,fuel_surcharge_pct,remote_surcharge,currency
0,6,express,0.0,0.5,5.6925,3.8775,0.18,0.0,USD
1,6,express,0.5,1.0,5.6925,3.8775,0.18,0.0,USD
2,6,express,1.0,2.0,5.6925,3.6836,0.18,0.0,USD
3,6,express,2.0,5.0,5.6925,3.4122,0.18,0.0,USD
4,6,express,5.0,10.0,5.6925,3.1020,0.18,0.0,USD
5,6,express,10.0,20.0,5.6925,2.7918,0.18,0.0,USD
6,6,express,20.0,30.0,5.6925,2.5592,0.18,0.0,USD
7,6,express,30.0,NaN,5.6925,2.3265,0.18,0.0,USD


---

## 先给好的中间结果：`priced`

R3 / R4 需要「按费率表算出应付金额」。这个**非等值 join** 比较机械，我先写好，
你把精力放在规则逻辑上。**（进阶挑战 1 会让你自己写它）**

```sql
LEFT JOIN rate AS r
  ON r.zone = i.dest_zone
  AND r.service_level = i.service_level
  AND i.chargeable_weight_kg >= r.weight_min_kg
  AND (r.weight_max_kg IS NULL OR i.chargeable_weight_kg < r.weight_max_kg)
```

> ⚠️ 最后一档 `weight_max_kg` 是 **NULL**（无上限），必须用 `IS NULL OR` 处理，
> 否则重量最大的那批运单会**匹配不上费率**、被静默漏掉。
>
> ⚠️ 金额全程用 `DECIMAL`，不用 `DOUBLE` —— 原因见 README「双实现交叉验证」
> （实测 float 路径有 **6% 的行**会算错）。

In [4]:
con.execute("""
CREATE OR REPLACE VIEW priced AS
WITH raw AS (
    SELECT
        i.shipment_id, i.ship_date, i.dest_zone, i.service_level, i.chargeable_weight_kg,
        CAST(r.base_rate          AS DECIMAL(18,4)) AS base_rate,
        CAST(r.per_kg_rate        AS DECIMAL(18,4)) AS per_kg_rate,
        CAST(r.fuel_surcharge_pct AS DECIMAL(18,4)) AS fuel_surcharge_pct,
        CAST(r.remote_surcharge   AS DECIMAL(18,4)) AS remote_surcharge,
        ROUND(CAST(r.per_kg_rate AS DECIMAL(18,4))
              * CAST(i.chargeable_weight_kg AS DECIMAL(18,3)), 4) AS weight_amount
    FROM internal AS i
    LEFT JOIN rate AS r
        ON  r.zone = i.dest_zone
        AND r.service_level = i.service_level
        AND i.chargeable_weight_kg >= r.weight_min_kg
        AND (r.weight_max_kg IS NULL OR i.chargeable_weight_kg < r.weight_max_kg)
),
stepped AS (SELECT *, ROUND(base_rate + weight_amount, 4) AS subtotal_before FROM raw),
fueled  AS (SELECT *, ROUND(subtotal_before * fuel_surcharge_pct, 4) AS fuel_amount FROM stepped)
SELECT
    shipment_id, ship_date, dest_zone, service_level, chargeable_weight_kg,
    base_rate, per_kg_rate, fuel_surcharge_pct, remote_surcharge,
    weight_amount, subtotal_before, fuel_amount,
    ROUND(subtotal_before + fuel_amount + remote_surcharge, 4) AS expected_total
FROM fueled
""")
print('✅ priced 视图已创建')
display(run('SELECT * FROM priced LIMIT 3'))

✅ priced 视图已创建


,shipment_id,ship_date,dest_zone,service_level,chargeable_weight_kg,base_rate,per_kg_rate,fuel_surcharge_pct,remote_surcharge,weight_amount,subtotal_before,fuel_amount,expected_total
0,SHP2026000985,2026-08-26,3,economy,0.254,2.1675,1.2325,0.13,0.0,0.3131,2.4806,0.3225,2.8031
1,SHP2026001720,2026-08-12,3,economy,0.290,2.1675,1.2325,0.13,0.0,0.3574,2.5249,0.3282,2.8531
2,SHP2026000564,2026-06-07,3,economy,0.603,2.1675,1.2325,0.13,0.0,0.7432,2.9107,0.3784,3.2891


---

# R1 · 未入账

**业务含义**：内部系统记录了发货，但承运商账单里**找不到这个运单**。

**为什么是问题**：可能是承运商漏开票（晚点会补开），也可能是我们的内部数据缺失。
不确认清楚，**应付账款的完整性就没保证**。

**输出要求**：至少包含 `shipment_id`（校验按这个字段比对）

In [7]:
# ✍️ R1：在这里写你的 SQL（把三引号里换成你的查询）
YOUR_SQL_R1 = """
/* 在这里写你的 SQL */
select internal.shipment_id
from internal left join invoice on internal.shipment_id = invoice.shipment_id
where invoice.shipment_id is NULL
"""

if _is_blank(YOUR_SQL_R1):
    print('⏳ 还没写 —— 上面的提示还不够的话，看下面「💡 卡住了再看」')
    df = None
else:
    df = run(YOUR_SQL_R1)
    print(f'你的结果：{len(df)} 条')
    display(df.head())

你的结果：184 条


,shipment_id
0,SHP2026001427
1,SHP2026003114
2,SHP2026006679
3,SHP2026006988
4,SHP2026009737


### 💡 卡住了再看：SQL 技巧提示

- 「左表有、右表无」→ **反连接**：`LEFT JOIN ... WHERE 右表主键 IS NULL`
- 也可以写 `NOT EXISTS`；`NOT IN` 要小心 NULL 的坑（想一想为什么）
- 表：`internal`（左）、`invoice`（右）；关联键 `shipment_id`

In [6]:
# ✅ 校验 R1
check('R1_MISSING_FROM_INVOICE', df)

参考条数   184   你的条数   184
  漏报     0   多报     0

✅ 完全一致，这条过了


True

---

# R2 · 重复计费

**业务含义**：同一运单在账单里出现了**多次**。业务约定是「一个运单 = 一条账单行」，
出现重复本身就是异常。

**处理原则**：按账单行号排序，**保留最早一行视为正常，其余算多收**。

**输出要求**：至少包含 `shipment_id`

In [8]:
# ✍️ R2：在这里写你的 SQL（把三引号里换成你的查询）
YOUR_SQL_R2 = """
/* 在这里写你的 SQL
with tmp as (
select *, row_number() over( partition by internal.shipment_id order by invoice_line_id ) as row_num
from internal
left join invoice on internal.shipment_id = invoice.shipment_id
)
select *
from tmp
where row_num > 1
 */

/*
QUALIFY 写法
*/
select *, row_number() over( partition by shipment_id order by invoice_line_id ) as row_num
from invoice
qualify row_number() over( partition by shipment_id order by invoice_line_id ) > 1
"""

if _is_blank(YOUR_SQL_R2):
    print('⏳ 还没写 —— 上面的提示还不够的话，看下面「💡 卡住了再看」')
    df = None
else:
    df = run(YOUR_SQL_R2)
    print(f'你的结果：{len(df)} 条')
    display(df.head())

你的结果：329 条


,invoice_line_id,invoice_batch,shipment_id,invoice_date,billed_zone,service_level,billed_weight_kg,base_amount,weight_amount,fuel_surcharge,remote_surcharge,total_amount,currency,row_num
0,L0001040,INV-202609-RE,SHP2026001025,2026-09-27,5,standard,0.569,3.20,1.1665,0.6550,0.0,5.0215,USD,2
1,L0002125,INV-202609-RE,SHP2026002097,2026-07-24,6,standard,0.963,3.45,2.2631,0.8570,0.0,6.5701,USD,2
2,L0006452,INV-202609-RE,SHP2026006396,2026-07-24,3,standard,1.989,2.55,2.7398,0.7935,0.0,6.0833,USD,2
3,L0006460,INV-202609-RE,SHP2026006403,2026-06-24,9,standard,0.511,4.50,1.7885,0.9433,2.8,10.0318,USD,2
4,L0006583,INV-202609-RE,SHP2026006523,2026-09-28,8,standard,1.484,4.15,4.3704,1.2781,2.2,11.9985,USD,2


### 💡 卡住了再看：SQL 技巧提示

- 找重复运单：`GROUP BY shipment_id HAVING COUNT(*) > 1`
- 取「除首行之外」的行：窗口函数 `ROW_NUMBER() OVER (PARTITION BY shipment_id ORDER BY invoice_line_id)`
- DuckDB 支持 `QUALIFY`，可以直接在窗口函数上过滤，不用再套一层子查询

**思考题**：为什么是「保留首行、其余算多收」，而不是「两条都算异常」？
（提示：想一想**金额**会不会被重复计算）

In [14]:
# ✅ 校验 R2
check('R2_DUPLICATE_BILLING', df)

参考条数   329   你的条数   329
  漏报     0   多报     0

✅ 完全一致，这条过了


True

---

# R3 · 计费重量偏差

**业务含义**：账单上的计费重量和我们的应付计费重量对不上。

**计费重量 = max(实际重量, 体积重)**，体积重 = 长×宽×高 / 6000。
承运商可能按体积重收费，或出现称重误差。

**判定条件（两个都要满足）**
```
相对偏差 > 5%        业务容忍度（通常写进合同 SLA）
且 绝对差 > 0.05 kg   抗称重噪声
```

**为什么要加绝对差下限**：对 0.05–0.3kg 的小件，5% 只有几克，这在真实称重里是噪声。
不加下限，小件会被频繁误报。**这个取舍后面有实验**（进阶挑战 2）。

**输出要求**：至少包含 `shipment_id`

In [9]:
# ✍️ R3：在这里写你的 SQL（把三引号里换成你的查询）
YOUR_SQL_R3 = """
/* 在这里写你的 SQL */
with tmp as (
select priced.shipment_id, row_number() over(partition by invoice.shipment_id order by invoice_line_id) as row_num, ABS(invoice.billed_weight_kg - priced.chargeable_weight_kg) as diff_weight, (invoice.billed_weight_kg - priced.chargeable_weight_kg) / priced.chargeable_weight_kg as diff_rate 
from priced left join invoice on priced.shipment_id = invoice.shipment_id
where invoice.shipment_id is not null and priced.chargeable_weight_kg > 0
)
select shipment_id
from tmp
where row_num = 1 and diff_weight > 0.05 and diff_rate > 0.05
"""
# YOUR_SQL_R3 = """
# show create table priced
# """

if _is_blank(YOUR_SQL_R3):
    print('⏳ 还没写 —— 上面的提示还不够的话，看下面「💡 卡住了再看」')
    df = None
else:
    df = run(YOUR_SQL_R3)
    print(f'你的结果：{len(df)} 条')
    display(df.head())

你的结果：755 条


,shipment_id
0,SHP2026000262
1,SHP2026000343
2,SHP2026000429
3,SHP2026000649
4,SHP2026001034


### 💡 卡住了再看：SQL 技巧提示

- 账单重量在 `invoice.billed_weight_kg`，应付重量在 `priced.chargeable_weight_kg`
- 相对偏差：`ABS(a - b) / b > 0.05`；别忘了 `b > 0` 防除零
- **暂时不用管重复计费的运单**，只要条数对得上就行

In [19]:
# ✅ 校验 R3
check('R3_WEIGHT_DEVIATION', df)

参考条数   755   你的条数   755
  漏报     0   多报     0

✅ 完全一致，这条过了


True

---

# R4 · 费率不符

**业务含义**：账单金额和**按费率表算出来的应付金额**对不上。分两种情形：

1. **分区用错**（最常见）：账单用了更高的分区计费
   → 直接比 `invoice.billed_zone` 和 `priced.dest_zone`
2. **费用项不符**：分区对，但某一项（基础费 / 续重费 / 燃油附加 / 偏远附加）不对
   → 逐项和 `priced` 里的对应列比，容差 1%

**输出要求**：至少包含 `shipment_id`

In [10]:
# ✍️ R4：在这里写你的 SQL（把三引号里换成你的查询）
YOUR_SQL_R4 = """
/* 在这里写你的 SQL */
with tmp as (
    select 
        priced.shipment_id, row_number() over(partition by invoice.shipment_id order by invoice_line_id) as row_num, ABS(invoice.billed_weight_kg - priced.chargeable_weight_kg) as diff_weight, (invoice.billed_weight_kg - priced.chargeable_weight_kg) / priced.chargeable_weight_kg as diff_rate,
        invoice.billed_zone, invoice.base_amount, invoice.weight_amount, invoice.fuel_surcharge, invoice.remote_surcharge
    from priced left join invoice on priced.shipment_id = invoice.shipment_id
    where invoice.shipment_id is not null and priced.chargeable_weight_kg > 0
), invoice_not_r3 as (
    select *
    from tmp
    where row_num = 1 and not (diff_weight > 0.05 and diff_rate > 0.05)
)
-- 分区不对
select p.shipment_id
from priced as p inner join invoice_not_r3 as i on p.shipment_id = i.shipment_id
where p.dest_zone != i.billed_zone
UNION
-- 分区对但是加和不对
select p.shipment_id
from priced as p inner join invoice_not_r3 as i on p.shipment_id = i.shipment_id
where p.dest_zone = i.billed_zone and (
-- 基础费
(p.base_rate > 0 and ABS( i.base_amount - p.base_rate ) / p.base_rate > 0.01)
OR
-- 续重费
(p.weight_amount > 0 and ABS( i.weight_amount - p.weight_amount ) / p.weight_amount > 0.01)
OR
-- 燃油附加
(p.fuel_amount > 0 and ABS( i.fuel_surcharge - p.fuel_amount ) / p.fuel_amount > 0.01)
OR
-- 偏远附加
(p.remote_surcharge > 0 and ABS( i.remote_surcharge - p.remote_surcharge ) / p.remote_surcharge > 0.01)
)
"""

if _is_blank(YOUR_SQL_R4):
    print('⏳ 还没写 —— 上面的提示还不够的话，看下面「💡 卡住了再看」')
    df = None
else:
    df = run(YOUR_SQL_R4)
    print(f'你的结果：{len(df)} 条')
    display(df.head())

你的结果：830 条


,shipment_id
0,SHP2026001477
1,SHP2026001522
2,SHP2026000420
3,SHP2026001572
4,SHP2026001337


### 💡 卡住了再看：SQL 技巧提示

- **先比分区**（最直接的证据），再逐项比金额
- 容差：`ABS(billed - expected) / expected > 0.01`，注意 `expected > 0` 防除零
- 两种情形都算 R4，用 `OR` 连起来（或者分两个查询再 UNION）

**取证小技巧**：如果账单金额**恰好等于另一个分区的应付**（一分不差），说明承运商不是算错，
而是**用错了分区去查费率表** —— 这个区别决定后续怎么跟对方交涉。

In [50]:
# ✅ 校验 R4
check('R4_RATE_MISMATCH', df)

参考条数   830   你的条数   830
  漏报     0   多报     0

✅ 完全一致，这条过了


True

---

# R5 · 金额加总错误

**业务含义**：账单行的**总额 ≠ 各项费用之和**。各项都对，就是加总错了 —— 账单生成逻辑的缺陷。

**输出要求**：至少包含 `shipment_id`

In [11]:
# ✍️ R5：在这里写你的 SQL（把三引号里换成你的查询）
YOUR_SQL_R5 = """

/* 在这里写你的 SQL */
with tmp as (
    select 
        priced.shipment_id, row_number() over(partition by invoice.shipment_id order by invoice_line_id) as row_num, ABS(invoice.billed_weight_kg - priced.chargeable_weight_kg) as diff_weight, (invoice.billed_weight_kg - priced.chargeable_weight_kg) / priced.chargeable_weight_kg as diff_rate,
        invoice.billed_zone, invoice.base_amount, invoice.weight_amount, invoice.fuel_surcharge, invoice.remote_surcharge,
        invoice.total_amount
    from priced left join invoice on priced.shipment_id = invoice.shipment_id
    where invoice.shipment_id is not null and priced.chargeable_weight_kg > 0
), invoice_not_r3 as (
    select *
    from tmp
    where row_num = 1 and not (diff_weight > 0.05 and diff_rate > 0.05)
)
select p.shipment_id
from priced as p inner join invoice_not_r3 as i on p.shipment_id = i.shipment_id
where p.dest_zone = i.billed_zone and not (
-- 基础费
(p.base_rate > 0 and ABS( i.base_amount - p.base_rate ) / p.base_rate > 0.01)
OR
-- 续重费
(p.weight_amount > 0 and ABS( i.weight_amount - p.weight_amount ) / p.weight_amount > 0.01)
OR
-- 燃油附加
(p.fuel_amount > 0 and ABS( i.fuel_surcharge - p.fuel_amount ) / p.fuel_amount > 0.01)
OR
-- 偏远附加
(p.remote_surcharge > 0 and ABS( i.remote_surcharge - p.remote_surcharge ) / p.remote_surcharge > 0.01)
)
-- R4 加总不对
and ABS(p.expected_total - i.total_amount ) > 0.05
"""

if _is_blank(YOUR_SQL_R5):
    print('⏳ 还没写 —— 上面的提示还不够的话，看下面「💡 卡住了再看」')
    df = None
else:
    df = run(YOUR_SQL_R5)
    print(f'你的结果：{len(df)} 条')
    display(df.head())

你的结果：405 条


,shipment_id
0,SHP2026000070
1,SHP2026000472
2,SHP2026000576
3,SHP2026001399
4,SHP2026001790


### 💡 卡住了再看：SQL 技巧提示

- `components = base_amount + weight_amount + fuel_surcharge + remote_surcharge`
- 判定：`ABS(total_amount - components) > 0.05`
- 这里用**绝对容差 0.05 美元**（分位），不是百分比 —— 因为是加总错误，不是费率错误

**⚠️ 关键设计**：这条必须在 R3、R4 都**不成立**时才判。
因为重量错、费率错也会导致总额不一致；否则同一笔差异会被**重复计算金额**。
（这叫「按根因优先级归类，避免一事多报」）

In [58]:
# ✅ 校验 R5
check('R5_AMOUNT_MISMATCH', df)

参考条数   405   你的条数   405
  漏报     0   多报     0

✅ 完全一致，这条过了


True

---

# 🚀 进阶挑战

## 挑战 1 · 自己写 `priced` 的非等值 join

把上面那个「先给好的」`priced` 抛开，自己写一遍费率匹配。

**自检标准**：结果应是 **20,000 行**，且 `matched` 也是 **20,000**（没有漏匹配、没有膨胀）。

In [61]:
# ✍️ 挑战 1：在这里写你的 SQL（把三引号里换成你的查询）
YOUR_SQL_JOIN = """
/* 在这里写你的 SQL：LEFT JOIN rate + 四个 ON 条件 + COUNT */
select internal.shipment_id
from internal left join rate on 
    internal.dest_zone = rate.zone 
    and internal.service_level = rate.service_level
    and internal.chargeable_weight_kg >= weight_min_kg
    and (weight_max_kg is NULL OR internal.chargeable_weight_kg < weight_max_kg)
"""

if _is_blank(YOUR_SQL_JOIN):
    print('⏳ 还没写 —— 上面的提示还不够的话，看下面「💡 卡住了再看」')
    df = None
else:
    df = run(YOUR_SQL_JOIN)
    print(f'你的结果：{len(df)} 条')
    display(df.head())

你的结果：20000 条


,shipment_id
0,SHP2026000985
1,SHP2026001720
2,SHP2026000564
3,SHP2026001604
4,SHP2026001776


### 💡 卡住了再看：要点提示

- 条件要保证**唯一命中**一个重量档 —— 写松了会产生一对多、**行数膨胀**
- 最后一档 `weight_max_kg` 是 NULL（无上限），别忘了处理
- 想看膨胀的话，把 `r.zone = i.dest_zone` 这个条件去掉试试，行数会炸

In [62]:
# 自检标准（不用改，跑一下看结论）
print('目标：rows = 20000, matched = 20000')
print('rows > 20000    → join 条件写松了，一对多膨胀')
print('matched < 20000 → 有运单没匹配上费率（想想是不是最后一档 NULL 没处理）')

目标：rows = 20000, matched = 20000
rows > 20000    → join 条件写松了，一对多膨胀
matched < 20000 → 有运单没匹配上费率（想想是不是最后一档 NULL 没处理）


---

## 挑战 2 · 阈值敏感性实验

这个实验就是 README 里那个 **75.6%** 的由来。改 R3 的绝对差下限，看条数怎么变：

| `min_delta` | 预期条数 |
| --- | --- |
| **0.05**（默认） | **755** |
| **0** | **997** |

**想一想**
- 为什么**去掉下限，条数反而变多**？
- 多出来的那 242 条，都是什么样的包裹？
- 如果这是真实生产系统，你会把下限设为多少？依据是什么？

In [5]:
# 这个不用你改，直接跑
for min_delta in (0.0, 0.05):
    n = run(f"""
        SELECT COUNT(*) AS n FROM invoice AS i JOIN priced AS p USING (shipment_id)
        WHERE ABS(i.billed_weight_kg - p.chargeable_weight_kg) / p.chargeable_weight_kg > 0.05
          AND ABS(i.billed_weight_kg - p.chargeable_weight_kg) > {min_delta}""").iloc[0, 0]
    print(f'min_delta={min_delta}  →  {n} 条')
print()
print('多出来的那批包裹有多重：')
display(run("""
    SELECT
      CASE WHEN p.chargeable_weight_kg < 0.5 THEN 'a. <0.5kg'
           WHEN p.chargeable_weight_kg < 1.0 THEN 'b. 0.5-1kg'
           WHEN p.chargeable_weight_kg < 3.0 THEN 'c. 1-3kg' ELSE 'd. >=3kg' END AS 重量段,
      COUNT(*) AS 条数_下限0,
      SUM(CASE WHEN ABS(i.billed_weight_kg - p.chargeable_weight_kg) > 0.05 THEN 1 ELSE 0 END) AS 条数_下限005
    FROM invoice AS i JOIN priced AS p USING (shipment_id)
    WHERE ABS(i.billed_weight_kg - p.chargeable_weight_kg) / p.chargeable_weight_kg > 0.05
    GROUP BY 1 ORDER BY 1"""))

min_delta=0.0  →  997 条
min_delta=0.05  →  755 条

多出来的那批包裹有多重：


,重量段,条数_下限0,条数_下限005
0,a. <0.5kg,265,57.0
1,b. 0.5-1kg,288,254.0
2,c. 1-3kg,307,307.0
3,d. >=3kg,137,137.0


---

## 挑战 3 · 交叉验证（⚠️ 这题和上面 5 条**不是一回事**）

### 先说清楚：这题**不需要**考虑互斥、**不需要**去重、**不需要**漏斗

上面 5 条规则是 **漏斗**：规则之间**互斥**，一个运单只能归到一个根因，
目的是**避免一事多报**（否则同一笔差异会被重复计算金额）。

这一题是 **交叉验证**：目的是确认「**你写的 SQL 和参考实现，抓到的运单是不是同一批**」。

| | 上面 5 条规则 | 挑战 3 |
| --- | --- | --- |
| 做什么 | 把 20,000 条运单**归类** | 把两份结果**比对** |
| 操作对象 | 原始数据 | 两套算法的输出 |
| 集合运算 | **差集**（逐步剔除） | **并集 + 对称差** |
| 要求互斥吗 | ✅ 必须 | ❌ 不需要 |
| 目的 | 归类根因 | 验证实现写对了没有 |

> 💡 比喻：上面 5 条是「**一条产线内部的工序**」，挑战 3 是「**两条产线比对成品**」。

### 要做什么（就三步）

1. 把 5 条规则的结果**取并集**（注意是并集，不是漏斗）
2. 和 `REFERENCE`（参考实现的结果）的 `shipment_id` 集合比对
3. 报出**漏报**（参考有、你没有）和**多报**（你有、参考没有）

```python
MINE = set().union(*[set(df.shipment_id) for df in [df_r1, df_r2, df_r3, df_r4, df_r5]])
REF  = set(REFERENCE.shipment_id)

print('漏报', len(REF - MINE), '  多报', len(MINE - REF))
print('不一致总数', len(REF ^ MINE))      # ^ 是对称差，一次表达「漏报 + 多报」
```

### 为什么用「对称差」而不是「逐步剔除」

如果你在这里做漏斗式的剔除（匹配上的就从集合里删掉），
**会把「归错类」这个错误本身也删掉** —— 那就什么都发现不了。

举个**你自己踩过的真实例子**：你之前那版 R4 没排除 R3 的运单，多报了 743 条。

| 比对方式 | 能得到的信息 |
| --- | --- |
| **按规则分别比对**（本题的做法） | 一眼看出「R4 多报 743」→ **能定位到规则** |
| 压成一个总数比对 | 只知道「多了 743 条」→ **不知道哪条规则错了** |

所以交叉验证必须**按规则逐条比对**，不能压扁。

### 目标

**漏报 0、多报 0** —— 和参考实现抓到完全同一批运单。

In [14]:
# ✍️ 挑战 3：把 5 条规则的结果取并集，和参考实现比对
#
# 说明：
#   · 这是 Python，不是 SQL
#   · 不需要建视图 —— 第 0 节已经建好了 internal / invoice / rate / priced
#   · 不需要剔除、不需要考虑互斥 —— 那是上面 5 条规则的漏斗逻辑，这题不用
#
# 变量名提示（重要）：
#   上面 5 个练习单元格用的都是同一个变量名 df，所以现在 df 只指向最后跑的那条。
#   两种解决办法（选一个）：
#     a) 回去把每个练习单元格末尾改成 df_r1 = run(...)、df_r2 = run(...) …
#     b) 或者直接在这里按顺序重跑一遍（见下面的注释）
#
import itertools

df_r1 = run(YOUR_SQL_R1)
df_r2 = run(YOUR_SQL_R2)
df_r3 = run(YOUR_SQL_R3)
df_r4 = run(YOUR_SQL_R4)
df_r5 = run(YOUR_SQL_R5)

MINE = set(itertools.chain(*[list(df.shipment_id) for df in [df_r1, df_r2, df_r3, df_r4, df_r5]]))      # ← 换成 5 个结果的并集（一个 set）

if MINE is None:
    print("⏳ 还没写 —— 把 5 个结果的 shipment_id 取并集赋给 MINE")
else:
    # 兼容两种写法：MINE 是 DataFrame 或已经是 set
    mine_ids = set(MINE.shipment_id) if hasattr(MINE, "shipment_id") else set(MINE)
    REF = set(REFERENCE.shipment_id)
    print(f"参考 {len(REF):,} 条   你的 {len(mine_ids):,} 条")
    print(f"  漏报 {len(REF - mine_ids)}   多报 {len(mine_ids - REF)}")
    print()
    if REF == mine_ids:
        print("✅ 完全一致 —— 你的 5 条规则和参考实现抓到的是同一批运单")
    else:
        print("❌ 还有差异 —— 回去看看是哪条规则多抓或少抓了")
        print(f"   漏报示例 {sorted(REF - mine_ids)[:3]}")
        print(f"   多报示例 {sorted(mine_ids - REF)[:3]}")

参考 2,503 条   你的 2,503 条
  漏报 0   多报 0

✅ 完全一致 —— 你的 5 条规则和参考实现抓到的是同一批运单


### 💡 卡住了再看：提示

### 第一步：把 5 个结果都拿到手

上面的练习单元格**都用了同一个变量名 `df`**，所以 `df` 现在只等于最后跑的那条规则。
先按顺序重跑一遍（或者回去把它们分别改名）：

```python
df_r1 = run(YOUR_SQL_R1); df_r2 = run(YOUR_SQL_R2); df_r3 = run(YOUR_SQL_R3)
df_r4 = run(YOUR_SQL_R4); df_r5 = run(YOUR_SQL_R5)
```

### 第二步：取并集（两种写法等价）

```python
from functools import reduce

# 函数式写法
MINE = reduce(set.union,
              (set(df.shipment_id) for df in [df_r1, df_r2, df_r3, df_r4, df_r5]),
              set())

# 或者更 Pythonic（* 解包）
MINE = set().union(*(set(df.shipment_id) for df in [df_r1, df_r2, df_r3, df_r4, df_r5]))
```

### 第三步：比对

```python
REF = set(REFERENCE.shipment_id)
print('漏报', len(REF - MINE), '  多报', len(MINE - REF))
print('不一致总数', len(REF ^ MINE))
```

> ⚠️ **这里没有 `NOT IN`、没有"从池子里剔除"** —— 那是上面 5 条规则的漏斗逻辑。
> 这题只是把两份结果叠起来看差，**别把漏斗思路带进来**。

---

# 📖 写完了？对照参考答案

完整参考实现在 **`sql/reconcile_duckdb.sql`**（329 行），包含：
- 5 条规则的完整 SQL，以及**根因优先级**的 `CASE` 写法
- 参数化阈值（`params` CTE）
- 严重度分档、`finding_id` 编号、汇总报告

## 最后检查一遍：这 5 点能讲清楚吗

| # | 要点 | 一句话 |
| --- | --- | --- |
| 1 | **应付口径唯一** | 账单生成和核对共用同一套计费逻辑，否则算出来的差异全是假的 |
| 2 | **按根因归类** | R3→R4→R5 优先级，避免一个错误被重复计算金额 |
| 3 | **精度/召回权衡** | 重量规则加绝对差下限抗噪，代价是小件根因归类率降到 75.6% |
| 4 | **金额必须用 DECIMAL** | float 路径 2 万行里 6% 结果错误（IEEE754 存不下 30.051945） |
| 5 | **双实现交叉验证** | Python + SQL 两套独立实现，32,539 项字段零差异 |

> 这 5 点是把这套规则讲清楚的骨架 —— 写完之后应该能不看代码复述出来。